In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

spark.sql("CREATE DATABASE IF NOT EXISTS silver")


DataFrame[]

## 1. silver.tb_info_filmes

Regras aplicadas:
- unicidade por filme, mantendo a ingestão mais recente;
- normalização e tradução do status;
- conversão robusta de datas em múltiplos formatos;
- criação de `ano_lancamento`;
- nomes de colunas em português e tipagem apropriada.


In [0]:
df_info_bronze = spark.table("bronze.tb_movies_info")

janela_filme = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc()
)

df_info_silver = (
    df_info_bronze
    .withColumn("rn", F.row_number().over(janela_filme))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_info_silver = df_info_silver.withColumn(
    "status_normalizado",
    F.initcap(
        F.trim(
            F.regexp_replace(
                F.regexp_replace(F.col("status"), r"[^A-Za-z\s-]", ""),
                r"[-\s]+",
                " "
            )
        )
    )
)

df_info_silver = df_info_silver.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "Released", "Lançado")
     .when(F.col("status_normalizado") == "Post Production", "Pós-Produção")
     .when(F.col("status_normalizado") == "In Production", "Em Produção")
     .when(F.col("status_normalizado") == "Planned", "Planejado")
     .when(F.col("status_normalizado") == "Rumored", "Rumores")
     .when(F.col("status_normalizado") == "Canceled", "Cancelado")
     .otherwise("Não Informado")
)

df_info_silver = df_info_silver.withColumn(
    "data_lancamento",
    F.coalesce(
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("MM/dd/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("yyyy/MM/dd")))
    )
)

df_info_silver = (
    df_info_silver
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        "data_lancamento",
        F.expr("try_cast(runtime as int)").alias("duracao_minutos"),
        F.col("original_language").alias("idioma_original"),
        "status_filme",
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao")
    )
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

(
    df_info_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

display(df_info_silver.limit(20))


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
14564,Rings,Rings,2017-02-01,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2017
32471,Mixtape,Mixtape,2021-12-03,94,en,Lançado,null,null,2021
38258,Grizzly II: Revenge,Grizzly II: Revenge,2020-02-17,74,en,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2020
38492,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,2022-06-22,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2022
38700,Bad Boys for Life,Bad Boys for Life,2020-01-15,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2020
42018,The Horse Thief,盗马贼,2019-03-19,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null,2019
42330,Monkey Magic,大闹西游,2018-09-22,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null,2018
43074,Ghostbusters,Ghostbusters,2016-07-14,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?,2016
45033,20 Seconds of Joy,20 Seconds of Joy,2018-01-01,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2018
46983,The Song of Styrene,Le Chant du styrène,2022-05-23,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null,2022


## 2. silver.tb_financeiro_filmes

Valores textuais inválidos são convertidos com segurança para `NULL`. Valores zero ou negativos também são invalidados. Para a conversão USD → BRL é utilizada a cotação de compra mais recente disponível na Bronze.


In [0]:
df_fin_bronze = spark.table("bronze.tb_movies_financials")

cotacao_atual = (
    spark.table("bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)

print("Cotação usada na conversão para BRL:", cotacao_atual)

df_fin_silver = (
    df_fin_bronze
    .withColumn(
        "orcamento_limpo",
        F.regexp_replace(F.col("budget").cast("string"), r"[^0-9\-]", "")
    )
    .withColumn(
        "receita_limpa",
        F.regexp_replace(F.col("revenue").cast("string"), r"[^0-9\-]", "")
    )
    .withColumn(
        "orcamento_usd",
        F.expr("try_cast(orcamento_limpo as decimal(18,2))")
    )
    .withColumn(
        "receita_usd",
        F.expr("try_cast(receita_limpa as decimal(18,2))")
    )
    .withColumn(
        "orcamento_usd",
        F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd"))
    )
    .withColumn(
        "receita_usd",
        F.when(F.col("receita_usd") > 0, F.col("receita_usd"))
    )
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(cotacao_atual)).cast(DecimalType(18, 2))
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(cotacao_atual)).cast(DecimalType(18, 2))
    )
    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd")).cast(DecimalType(18, 2))
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl")).cast(DecimalType(18, 2))
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull() & (F.col("receita_usd") != 0),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)
        )
    )
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual"
    )
)

(
    df_fin_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_financeiro_filmes")
)

display(df_fin_silver.limit(20))


Cotação usada na conversão para BRL: 5.1569


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,299100200.00,null,null,null,null
299536,300000000.00,2052415039.00,1547070000.00,10584099114.62,1752415039.00,9037029114.62,85.38
299534,356000000.00,2800000000.00,1835856400.00,14439320000.00,2444000000.00,12603463600.00,87.29
475557,55000000.00,1074458282.00,283629500.00,5540873914.45,1019458282.00,5257244414.45,94.88
271110,250000000.00,null,1289225000.00,null,null,null,null
284054,200000000.00,1349926083.00,1031380000.00,6961433817.42,1149926083.00,5930053817.42,85.18
284052,180000000.00,676343174.00,928242000.00,3487834114.00,496343174.00,2559592114.00,73.39
315635,175000000.00,880166924.00,902457500.00,4538932810.38,705166924.00,3636475310.38,80.12
283995,200000000.00,863756051.00,1031380000.00,4454303579.40,663756051.00,3422923579.40,76.85
297761,175000000.00,746846894.00,902457500.00,3851414747.67,571846894.00,2948957247.67,76.57


## 3. silver.tb_metricas_engajamento

A conversão usa `try_cast` para impedir que textos deslocados ou caracteres incompatíveis interrompam o pipeline. Notas fora de 0–10 e valores negativos são tratados como `NULL`.


In [0]:
df_metricas_bronze = spark.table("bronze.tb_movies_metrics")

def normalizar_decimal(coluna):
    texto = F.trim(
        F.regexp_replace(
            F.col(coluna).cast("string"),
            r"[^0-9,.\-]",
            ""
        )
    )

    formato_br = F.regexp_replace(
        F.regexp_replace(texto, r"\.", ""),
        ",",
        "."
    )

    formato_us = F.regexp_replace(texto, ",", "")
    somente_virgula = F.regexp_replace(texto, ",", ".")

    return (
        F.when(
            texto.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
            formato_br
        )
        .when(
            texto.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
            formato_us
        )
        .when(
            texto.rlike(r"^-?\d+,\d+$"),
            somente_virgula
        )
        .otherwise(texto)
    )

df_metricas_silver = (
    df_metricas_bronze
    .withColumn("popularidade_tmp", normalizar_decimal("popularity"))
    .withColumn("popularidade", F.expr("try_cast(popularidade_tmp as double)"))
    .withColumn("nota_tmdb_tmp", normalizar_decimal("vote_average"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(nota_tmdb_tmp as double)"))
    .withColumn("nota_imdb_tmp", normalizar_decimal("averageRating"))
    .withColumn("nota_media_imdb", F.expr("try_cast(nota_imdb_tmp as double)"))
    .withColumn(
        "qtd_votos_tmdb_tmp",
        F.regexp_replace(F.col("vote_count").cast("string"), ",", "")
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.expr("try_cast(qtd_votos_tmdb_tmp as int)")
    )
    .withColumn(
        "qtd_votos_imdb_tmp",
        F.regexp_replace(F.col("numVotes").cast("string"), ",", "")
    )
    .withColumn(
        "qtd_votos_imdb",
        F.expr("try_cast(qtd_votos_imdb_tmp as int)")
    )
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") >= 0, F.col("popularidade"))
    )
    .withColumn(
        "nota_media_tmdb",
        F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb"))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb"))
    )
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

(
    df_metricas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_metricas_engajamento")
)

display(df_metricas_silver.limit(20))


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


## 4. silver.tb_avaliacoes_usuarios

São removidos registros integralmente duplicados. Notas fora da escala 0–10 viram `NULL`, e comentários nulos ou em branco recebem `"Sem comentário"`.


In [0]:
df_reviews_bronze = spark.table("bronze.tb_movies_reviews")

df_avaliacoes_silver = (
    df_reviews_bronze
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .withColumn("nota_usuario_tmp", F.expr("try_cast(nota as double)"))
    .withColumn(
        "nota_usuario",
        F.when(
            F.col("nota_usuario_tmp").between(0, 10),
            F.col("nota_usuario_tmp")
        )
    )
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""),
            F.lit("Sem comentário")
        ).otherwise(F.trim(F.col("comentario")))
    )
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("nome").alias("nome_usuario"),
        "nota_usuario",
        "comentario_usuario"
    )
)

(
    df_avaliacoes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_avaliacoes_usuarios")
)

display(df_avaliacoes_silver.limit(20))


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


## 5. silver.tb_generos

A coluna de gêneros é normalizada para aceitar vírgula e ponto e vírgula, depois é desmembrada com `split + explode`. Valores vazios e resíduos puramente numéricos são removidos.


In [0]:
df_credits_bronze = spark.table("bronze.tb_credits_and_tags")

df_generos_silver = (
    df_credits_bronze
    .withColumn(
        "genres_normalizado",
        F.regexp_replace(F.col("genres"), ";", ",")
    )
    .withColumn(
        "genero",
        F.explode(F.split(F.col("genres_normalizado"), ","))
    )
    .withColumn("genero", F.trim(F.col("genero")))
    .filter(F.col("genero").isNotNull() & (F.col("genero") != ""))
    .filter(~F.col("genero").rlike(r"^[0-9.\-]+$"))
    .withColumn("genero", F.initcap(F.col("genero")))
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "genero"
    )
    .dropDuplicates(["id_filme", "genero"])
)

(
    df_generos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_generos")
)

display(
    df_generos_silver
    .groupBy("genero")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)


genero,count
Drama,30834
Documentary,18860
Comedy,17570
Thriller,9547
Horror,9287
Romance,7075
Action,5598
Crime,4365
Animation,4214
Tv Movie,3714


## 6. silver.tb_pessoas_empresas

`cast`, `directors`, `writers` e `production_companies` são consolidados em uma única tabela com os tipos `Ator`, `Diretor`, `Roteirista` e `Produtora`.


In [0]:
df_credits_bronze = spark.table("bronze.tb_credits_and_tags")

def explodir_entidade(coluna_origem, tipo_entidade):
    return (
        df_credits_bronze
        .withColumn(
            "valor_normalizado",
            F.regexp_replace(F.col(coluna_origem), ";", ",")
        )
        .withColumn(
            "nome_entidade",
            F.explode(F.split(F.col("valor_normalizado"), ","))
        )
        .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
        .filter(
            F.col("nome_entidade").isNotNull() &
            (F.col("nome_entidade") != "")
        )
        .filter(~F.col("nome_entidade").rlike(r"^[0-9.\-]+$"))
        .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
        .select(
            F.col("id").cast("string").alias("id_filme"),
            "nome_entidade",
            F.lit(tipo_entidade).alias("tipo_entidade")
        )
    )

df_atores = explodir_entidade("cast", "Ator")
df_diretores = explodir_entidade("directors", "Diretor")
df_roteiristas = explodir_entidade("writers", "Roteirista")
df_produtoras = explodir_entidade("production_companies", "Produtora")

df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

(
    df_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_pessoas_empresas")
)

display(
    df_pessoas_empresas
    .groupBy("tipo_entidade")
    .count()
)


tipo_entidade,count
Diretor,109138
Roteirista,138705
Produtora,120254
Ator,547252


## 7. silver.tb_cotacao_dolar

É gerado um calendário diário contínuo entre a primeira e a última cotação disponíveis. Dias sem cotação recebem o último valor conhecido por Forward Fill.


In [0]:
df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")

df_cotacoes = (
    df_cotacao_bronze
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .select(
        "data_cotacao",
        F.col("cotacaoCompra").cast("double").alias("cotacao_compra")
    )
    .dropDuplicates(["data_cotacao"])
)

limites = df_cotacoes.agg(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).first()

data_min = limites["data_min"]
data_max = limites["data_max"]

df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_min),
                F.lit(data_max),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

df_cotacao_silver = (
    df_calendario
    .join(df_cotacoes, on="data_cotacao", how="left")
)

janela_cotacao = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    df_cotacao_silver
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(janela_cotacao)
    )
    .orderBy("data_cotacao")
)

(
    df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_cotacao_dolar")
)

display(df_cotacao_silver)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-11,5.0912
2026-09-12,5.0912
2026-09-13,5.0912
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569


## Validação final da camada Silver


In [0]:
tabelas_silver = [
    "silver.tb_info_filmes",
    "silver.tb_financeiro_filmes",
    "silver.tb_metricas_engajamento",
    "silver.tb_avaliacoes_usuarios",
    "silver.tb_generos",
    "silver.tb_pessoas_empresas",
    "silver.tb_cotacao_dolar"
]

for tabela in tabelas_silver:
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

notas_invalidas = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .filter((F.col("nota_usuario") < 0) | (F.col("nota_usuario") > 10))
    .count()
)

print("Notas de usuário fora de 0–10:", notas_invalidas)

print("Tipos de entidade:")
display(
    spark.table("silver.tb_pessoas_empresas")
    .groupBy("tipo_entidade")
    .count()
)

print("Cotação Silver com Forward Fill:")
display(
    spark.table("silver.tb_cotacao_dolar")
    .orderBy("data_cotacao")
)


silver.tb_info_filmes: 97879 linhas
silver.tb_financeiro_filmes: 106165 linhas
silver.tb_metricas_engajamento: 107364 linhas
silver.tb_avaliacoes_usuarios: 32412 linhas
silver.tb_generos: 139471 linhas
silver.tb_pessoas_empresas: 915349 linhas
silver.tb_cotacao_dolar: 8 linhas
Notas de usuário fora de 0–10: 0
Tipos de entidade:


tipo_entidade,count
Ator,547252
Diretor,109138
Roteirista,138705
Produtora,120254


Cotação Silver com Forward Fill:


data_cotacao,cotacao_compra
2026-09-11,5.0912
2026-09-12,5.0912
2026-09-13,5.0912
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569
